# ARC-AGI-3 Research Agent — LoRA intervention (2026-08-18)

**Intervention:** Bootstrap Kaggle notebook from ASRA Phase 7 with HypothesisLoRA cache bridge; export cycle transition JSONL to ASRA-LoRA datasets; retrain adapters on merged corpora.

**ExplorationLoRA plan:** ACTION2, ACTION3, Log transition JSONL for LoRA cache refresh, Fix gateway/runtime errors before exploration policy changes

**Strategy:** see `research/strategies/2026-08-18/next-submission-plan.md`

HypothesisLoRA cache packaged from `research/adapters/` when available.


# ASRA Phase 4 — ARC Prize 2026

Built from `asra_phase4_kaggle_template_agent.py` via `_shared/gateway_notebook.py`. Official ARC-AGI-3 gateway sidecar + dummy parquet gate.

**Agent tag:** `asra-v0.6-phase4`

In [ ]:
!pip install --no-index --find-links \
 /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
 arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
"""ASRA Phase 4 agent — Kaggle template form (auto-extracted).

Spliced into submission notebook. Must expose class MyAgent.
"""

from __future__ import annotations

import hashlib
import json
import os
import random
from collections import Counter, defaultdict, deque
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

# --- Phase 3 compact exploration hints (embedded) ---


class CompactExplorationHints:
    """Visit memory + frontier bonus for action scoring."""

    def __init__(self, recent_window: int = 20) -> None:
        self.visit_counts: Dict[str, int] = defaultdict(int)
        self.recent: deque[str] = deque(maxlen=recent_window)
        self.edge_stats: Dict[Tuple[str, str], Dict[str, float]] = defaultdict(
            lambda: {"novelty": 0.0, "usefulness": 0.0, "n": 0.0}
        )

    def observe(self, state_hash: str, next_hash: str, action: str, reward: float, changed_cells: int) -> None:
        self.visit_counts[state_hash] += 1
        self.visit_counts[next_hash] += 1
        self.recent.append(next_hash)
        novelty = 1.0 / (1.0 + self.visit_counts[next_hash]) ** 0.5
        usefulness = float(reward) + (0.5 if changed_cells else 0.0)
        key = (state_hash, action)
        stats = self.edge_stats[key]
        stats["n"] += 1
        stats["novelty"] = ((stats["novelty"] * (stats["n"] - 1)) + novelty) / stats["n"]
        stats["usefulness"] = ((stats["usefulness"] * (stats["n"] - 1)) + usefulness) / stats["n"]

    def score_action(self, state_hash: str, action: str) -> float:
        visits = self.visit_counts[state_hash]
        frontier_bonus = 1.0 / (1.0 + visits) ** 0.5
        stats = self.edge_stats.get((state_hash, action))
        if stats and stats["n"] > 0:
            repeat_penalty = 1.0 if self.recent.count(state_hash) > 2 else 0.0
            return stats["novelty"] + stats["usefulness"] + frontier_bonus - repeat_penalty
        return frontier_bonus + 0.5

    def exploration_metadata(self, state_hash: str) -> Dict[str, Any]:
        return {
            "visit_count_before": self.visit_counts[state_hash],
            "frontier_node": self.visit_counts[state_hash] <= 2,
            "recent_unique": len(set(self.recent)),
        }


# --- Phase 2 compact object perception (embedded) ---


def _dominant_background(grid: List[List[int]]) -> int:
    counts: Counter[int] = Counter()
    for row in grid:
        counts.update(row)
    return counts.most_common(1)[0][0] if counts else 0


def _connected_components(
    grid: List[List[int]], background: int, connectivity: int = 4
) -> List[Tuple[int, List[Tuple[int, int]]]]:
    h = len(grid)
    w = len(grid[0]) if h else 0
    visited = [[False] * w for _ in range(h)]
    components: List[Tuple[int, List[Tuple[int, int]]]] = []
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)] if connectivity == 4 else [
        (-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)
    ]
    for y in range(h):
        for x in range(w):
            color = grid[y][x]
            if color == background or visited[y][x]:
                continue
            stack = [(y, x)]
            visited[y][x] = True
            pixels: List[Tuple[int, int]] = []
            while stack:
                cy, cx = stack.pop()
                pixels.append((cy, cx))
                for dy, dx in neighbors:
                    ny, nx = cy + dy, cx + dx
                    if 0 <= ny < h and 0 <= nx < w and not visited[ny][nx] and grid[ny][nx] == color:
                        visited[ny][nx] = True
                        stack.append((ny, nx))
            components.append((color, pixels))
    return components


def compact_scene(grid: List[List[int]]) -> Dict[str, Any]:
    if not grid or not grid[0]:
        return {"grid_shape": [0, 0], "background_color": 0, "num_objects": 0, "objects": []}
    bg = _dominant_background(grid)
    objects = []
    for idx, (color, pixels) in enumerate(_connected_components(grid, bg)):
        if not pixels:
            continue
        ys = [p[0] for p in pixels]
        xs = [p[1] for p in pixels]
        bbox = [min(ys), min(xs), max(ys), max(xs)]
        n = len(pixels)
        objects.append(
            {
                "object_id": f"obj_{idx}",
                "color": int(color),
                "area": n,
                "bbox": bbox,
                "centroid": [sum(ys) / n, sum(xs) / n],
            }
        )
    return {
        "grid_shape": [len(grid), len(grid[0])],
        "background_color": int(bg),
        "num_objects": len(objects),
        "objects": objects,
    }


def object_delta(before: Dict[str, Any], after: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "delta_num_objects": int(after.get("num_objects", 0)) - int(before.get("num_objects", 0)),
        "before_num_objects": int(before.get("num_objects", 0)),
        "after_num_objects": int(after.get("num_objects", 0)),
    }


def transform_histogram_from_scenes(before: Dict[str, Any], after: Dict[str, Any]) -> Dict[str, int]:
    """Lightweight transform tags from object-scene deltas (Phase 4 embedded)."""
    hist: Counter = Counter()
    delta = int(after.get("num_objects", 0)) - int(before.get("num_objects", 0))
    if delta > 0:
        hist["create"] += delta
    elif delta < 0:
        hist["delete"] += -delta
    before_objs = {o["object_id"]: o for o in before.get("objects", [])}
    after_objs = {o["object_id"]: o for o in after.get("objects", [])}
    for oid, obj_b in before_objs.items():
        obj_a = after_objs.get(oid)
        if obj_a is None:
            continue
        if obj_b.get("color") != obj_a.get("color"):
            hist["recolor"] += 1
        cy0, cx0 = obj_b.get("centroid", [0, 0])
        cy1, cx1 = obj_a.get("centroid", [0, 0])
        if abs(cy1 - cy0) > 0.4 or abs(cx1 - cx0) > 0.4:
            hist["translate"] += 1
    if not hist and before.get("num_objects") == after.get("num_objects"):
        hist["identity"] = 1
    return dict(hist)


# --- Phase 4 causal semantics (embedded) ---


class CausalSemanticsEngine:
    """Effect signatures, prediction, uncertainty, counterfactual."""

    def __init__(self) -> None:
        self.effects: Dict[Tuple[str, str], List[Dict[str, Any]]] = defaultdict(list)
        self.successors: Dict[Tuple[str, str], Counter] = defaultdict(Counter)

    def observe(
        self,
        state_hash_value: str,
        action: str,
        diff: Dict[str, Any],
        reward: float,
        *,
        next_hash: Optional[str] = None,
    ) -> None:
        self.effects[(state_hash_value, action)].append(
            {
                "num_changed_cells": diff.get("num_changed_cells"),
                "reward": reward,
                "delta_num_objects": diff.get("delta_num_objects"),
                "transform_histogram": dict(diff.get("transform_histogram") or {}),
                "next_hash": next_hash,
            }
        )
        if next_hash:
            self.successors[(state_hash_value, action)][next_hash] += 1

    def _label(self, mean: float, obj_mean: float, hist: Dict[str, int]) -> str:
        if mean == 0 and obj_mean == 0:
            return "no_op"
        top = max(hist, key=hist.get) if hist else None
        if top == "translate":
            return "translate"
        if top == "recolor":
            return "recolor"
        if top == "create":
            return "create_object"
        if top == "delete":
            return "delete_object"
        if mean <= 1.5:
            return "localized_transform"
        if obj_mean != 0:
            return "object_count_change"
        return "multi_cell_transform"

    def infer(self, state_hash_value: str, action: str) -> Dict[str, Any]:
        effects = self.effects.get((state_hash_value, action), [])
        if not effects:
            return {
                "observations": 0,
                "semantic_label": "unknown",
                "hypothesis": "unknown",
                "consistency_score": None,
                "confidence": 0.0,
                "uncertainty": 1.0,
                "predicted_changed_cells": 0.0,
            }
        counts = [float(e["num_changed_cells"]) for e in effects if e["num_changed_cells"] is not None]
        obj_deltas = [float(e.get("delta_num_objects") or 0) for e in effects]
        std = float(np.std(counts)) if counts else 0.0
        mean = float(np.mean(counts)) if counts else 0.0
        obj_mean = float(np.mean(obj_deltas)) if obj_deltas else 0.0
        hist: Counter = Counter()
        for e in effects:
            hist.update(e.get("transform_histogram") or {})
        label = self._label(mean, obj_mean, dict(hist))
        consistency = float(1.0 / (1.0 + std)) if counts else 0.0
        n = len(effects)
        confidence = min(1.0, (n / 5.0) * 0.6 + consistency * 0.4)
        uncertainty = min(1.0, (1.0 / (1.0 + n) ** 0.5) + 0.15 * min(1.0, std / max(1.0, mean + 1.0)))
        return {
            "observations": n,
            "semantic_label": label,
            "hypothesis": label,
            "consistency_score": consistency,
            "confidence": confidence,
            "uncertainty": uncertainty,
            "mean_delta_objects": obj_mean,
            "transform_histogram": dict(hist),
            "predicted_changed_cells": mean,
        }

    def predict_next(self, state_hash_value: str, action: str) -> Dict[str, Any]:
        counts = self.successors.get((state_hash_value, action))
        if not counts:
            return {"next_hash": None, "probability": 0.0}
        total = sum(counts.values())
        next_hash, top = counts.most_common(1)[0]
        return {"next_hash": next_hash, "probability": top / total}

    def counterfactual(self, state_hash_value: str, actual_action: str, alt_action: str) -> Dict[str, Any]:
        sem = self.infer(state_hash_value, alt_action)
        pred = self.predict_next(state_hash_value, alt_action)
        return {
            "actual_action": actual_action,
            "alt_action": alt_action,
            "predicted_changed_cells": sem.get("predicted_changed_cells", 0.0),
            "semantic_label": sem.get("semantic_label", "unknown"),
            "confidence": sem.get("confidence", 0.0),
            "next_hash": pred.get("next_hash"),
            "probability": pred.get("probability", 0.0),
        }


from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState


SEED = int(os.environ.get("ASRA_SEED", "42"))
MAX_ACTIONS = int(os.environ.get("ASRA_MAX_ACTIONS", "80"))
SIMPLE_ACTIONS = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION7"]
OBJECT_HINT_WEIGHT = float(os.environ.get("ASRA_OBJECT_HINT_WEIGHT", "0.30"))
EXPLORATION_HINT_WEIGHT = float(os.environ.get("ASRA_EXPLORATION_HINT_WEIGHT", "0.35"))
SEMANTICS_HINT_WEIGHT = float(os.environ.get("ASRA_SEMANTICS_HINT_WEIGHT", "0.35"))
PREDICTION_HINT_WEIGHT = float(os.environ.get("ASRA_PREDICTION_HINT_WEIGHT", "0.20"))
UNCERTAINTY_HINT_WEIGHT = float(os.environ.get("ASRA_UNCERTAINTY_HINT_WEIGHT", "0.30"))

random.seed(SEED)
np.random.seed(SEED)


def canonical_grid(grid: Any) -> List[List[int]]:
    return np.array(grid, dtype=int).tolist()


def state_hash(grid: Any) -> str:
    payload = json.dumps(canonical_grid(grid), separators=(",", ":"), sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


class ASRAExplorer:
    """Phase 2 object hints + Phase 3 exploration + Phase 4 causal semantics."""

    def __init__(self, action_names: List[str]) -> None:
        self.action_names = action_names
        self.state_action_counts: Counter = Counter()
        self.action_rewards: Dict[str, List[float]] = defaultdict(list)
        self.object_effect_scores: Dict[Tuple[str, str], float] = defaultdict(float)
        self.dead_ends: set = set()
        self.exploration = CompactExplorationHints()

    def update(
        self,
        state_hash_value: str,
        next_hash: str,
        action: str,
        diff: Dict[str, Any],
        reward: float,
    ) -> None:
        self.state_action_counts[(state_hash_value, action)] += 1
        self.action_rewards[action].append(float(reward))
        changed = int(diff.get("num_changed_cells") or 0)
        if changed == 0 and reward <= 0:
            self.dead_ends.add((state_hash_value, action))
        delta_obj = diff.get("delta_num_objects")
        if delta_obj is not None and delta_obj != 0:
            self.object_effect_scores[(state_hash_value, action)] += 0.5 * (1.0 if delta_obj > 0 else 0.7)
        self.exploration.observe(state_hash_value, next_hash, action, reward, changed)

    def choose_action(
        self,
        state_hash_value: str,
        semantics: CausalSemanticsEngine,
        available: Optional[List[str]] = None,
        scene_hint: Optional[Dict[str, Any]] = None,
    ) -> Tuple[str, Dict[str, Any]]:
        candidates = [a for a in self.action_names if available is None or a in available] or list(self.action_names)
        scores: Dict[str, float] = {}
        meta_by_action: Dict[str, Dict[str, Any]] = {}
        num_objects = int((scene_hint or {}).get("num_objects", 0))
        for action in candidates:
            if (state_hash_value, action) in self.dead_ends:
                continue
            sem = semantics.infer(state_hash_value, action)
            uncertainty = float(sem.get("uncertainty") or 1.0)
            confidence = float(sem.get("confidence") or 0.0)
            predicted = float(sem.get("predicted_changed_cells") or 0.0)
            local = self.state_action_counts[(state_hash_value, action)]
            mean_r = float(np.mean(self.action_rewards[action])) if self.action_rewards[action] else 0.0
            obj_bonus = OBJECT_HINT_WEIGHT * self.object_effect_scores.get((state_hash_value, action), 0.0)
            if num_objects == 0:
                obj_bonus *= 0.5
            explore_bonus = EXPLORATION_HINT_WEIGHT * self.exploration.score_action(state_hash_value, action)
            sem_bonus = SEMANTICS_HINT_WEIGHT * confidence
            pred_bonus = PREDICTION_HINT_WEIGHT * min(1.0, predicted / 10.0)
            unc_bonus = UNCERTAINTY_HINT_WEIGHT * uncertainty
            scores[action] = (
                2.0 / (1.0 + local)
                + unc_bonus
                + sem_bonus
                + pred_bonus
                + 0.5 * mean_r
                + obj_bonus
                + explore_bonus
                + random.random() * 0.05
            )
            meta_by_action[action] = sem
        if not scores:
            action = random.choice(candidates)
            return action, meta_by_action.get(action, {})
        action = max(scores.items(), key=lambda kv: kv[1])[0]
        return action, meta_by_action.get(action, {})


GLOBAL_SEMANTICS = CausalSemanticsEngine()
GLOBAL_EXPLORER = ASRAExplorer(SIMPLE_ACTIONS)


class MyAgent(Agent):
    """Phase 4: object scenes + exploration memory + causal semantics hints."""

    MAX_ACTIONS = MAX_ACTIONS

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN or self.action_counter >= self.MAX_ACTIONS

    def _available_simple(self, latest_frame: FrameData) -> List[str]:
        avail = getattr(latest_frame, "available_actions", None) or []
        names = [a.name for a in avail if hasattr(a, "name") and a.name in SIMPLE_ACTIONS]
        return names or SIMPLE_ACTIONS

    def _to_game_action(
        self,
        action_name: str,
        grid: Any,
        scene_hint: Optional[Dict[str, Any]] = None,
        causality_meta: Optional[Dict[str, Any]] = None,
    ) -> GameAction:
        ga = getattr(GameAction, action_name)
        if ga.is_complex():
            h, w = len(grid), len(grid[0]) if grid else 0
            ga.set_data({"x": w // 2, "y": h // 2})
        sh = state_hash(grid)
        explore_meta = GLOBAL_EXPLORER.exploration.exploration_metadata(sh)
        n_obj = (scene_hint or {}).get("num_objects", "?")
        sem = (causality_meta or {}).get("semantic_label", "unknown")
        conf = (causality_meta or {}).get("confidence", 0.0)
        unc = (causality_meta or {}).get("uncertainty", 1.0)
        ga.reasoning = (
            f"ASRA Phase4: {action_name} | objects={n_obj} "
            f"| visits={explore_meta.get('visit_count_before', 0)} "
            f"| sem={sem} conf={conf:.2f} u={unc:.2f}"
        )
        return ga

    def choose_action(self, frames: List[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        grid = latest_frame.frame
        scene = compact_scene(canonical_grid(grid))
        name, causality_meta = GLOBAL_EXPLORER.choose_action(
            state_hash(grid), GLOBAL_SEMANTICS, self._available_simple(latest_frame), scene_hint=scene
        )
        return self._to_game_action(name, grid, scene, causality_meta)

    def take_action(self, action: GameAction) -> Optional[FrameData]:
        self._last_action_name = action.name
        return super().take_action(action)

    def append_frame(self, frame: FrameData) -> None:
        super().append_frame(frame)
        if len(self.frames) < 2:
            return
        prev, curr = self.frames[-2], self.frames[-1]
        if not prev.frame or not curr.frame:
            return
        prev_grid = canonical_grid(prev.frame)
        curr_grid = canonical_grid(curr.frame)
        diff_count = int(np.sum(np.array(prev_grid) != np.array(curr_grid)))
        scene_before = compact_scene(prev_grid)
        scene_after = compact_scene(curr_grid)
        transform_hist = transform_histogram_from_scenes(scene_before, scene_after)
        diff = {
            "num_changed_cells": diff_count,
            "object_scene_before": scene_before,
            "object_scene_after": scene_after,
            "transform_histogram": transform_hist,
            **object_delta(scene_before, scene_after),
        }
        reward = float(getattr(curr, "levels_completed", 0) or 0)
        sh = state_hash(prev_grid)
        nsh = state_hash(curr_grid)
        action = getattr(self, "_last_action_name", "UNKNOWN")
        GLOBAL_SEMANTICS.observe(sh, action, diff, reward, next_hash=nsh)
        GLOBAL_EXPLORER.update(sh, nsh, action, diff, reward)


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
    --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
    /kaggle/working/ARC-AGI-3-Agents

    !cp /tmp/my_agent.py \
    /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
    MPLBACKEND=agg \
    python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
